<a href="https://colab.research.google.com/github/dangbenzy/Projects/blob/main/Transfer_learning_incomplete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Image Classification of Bird Species using a pretrained model

In [1]:
!pip install torchvision -q

In [2]:
# Import torch and torchvision modules
from torchvision.models import resnet50, ResNet50_Weights # To load any classification model.
from PIL import Image, ImageDraw, ImageFont # To read images from disk.
from torchvision import transforms, datasets, models # To apply PyTorch transformations
from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm


import os
import random
import requests # To download file.
import cv2 # For annotating images.
import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt # To visualize images.
from zipfile import ZipFile
from urllib.request import urlretrieve
!pip install torchinfo -q
from torchinfo import summary

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dmcgow/birds-200")

print("Path to dataset files:", path)

100%|██████████| 1.06G/1.06G [00:11<00:00, 101MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dmcgow/birds-200/versions/2


In [4]:
# Specify image transformations.
transform = ResNet50_Weights.IMAGENET1K_V2.transforms()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [31]:
train_path = path + "/birds_train"
test_path = path + "/birds_test"
dataset_train = datasets.ImageFolder(train_path, transform=transform)

batch_size = 64
num_of_epoch = 10
# Number of classes
num_classes = len(dataset_train.classes)  #10#2#257
print(num_classes)


train_data_loader= DataLoader(dataset_train, batch_size=32, shuffle=True)


200


In [6]:
print('num of training images:', len(dataset_train))
print('num of test images: ', len(test_path) )

num of training images: 8000
num of test images:  70


In [7]:
# Load pretrained ResNet50 Model
resnet50 = models.resnet50(weights='DEFAULT')
resnet50 = resnet50.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 130MB/s]


In [32]:
# Freeze model parameters
for param in resnet50.parameters():
    param.requires_grad = False

In [33]:

resnet50 = models.resnet50(weights='DEFAULT')


for param in resnet50.parameters():
    param.requires_grad = False

fc_inputs = resnet50.fc.in_features
resnet50.fc = nn.Sequential(
    nn.Linear(fc_inputs, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, num_classes),
    nn.LogSoftmax(dim=1)
)
resnet50 = resnet50.to(device)

print(summary(resnet50, input_size=(batch_size, 3, 224, 224)))

Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [64, 200]                 --
├─Conv2d: 1-1                            [64, 64, 112, 112]        (9,408)
├─BatchNorm2d: 1-2                       [64, 64, 112, 112]        (128)
├─ReLU: 1-3                              [64, 64, 112, 112]        --
├─MaxPool2d: 1-4                         [64, 64, 56, 56]          --
├─Sequential: 1-5                        [64, 256, 56, 56]         --
│    └─Bottleneck: 2-1                   [64, 256, 56, 56]         --
│    │    └─Conv2d: 3-1                  [64, 64, 56, 56]          (4,096)
│    │    └─BatchNorm2d: 3-2             [64, 64, 56, 56]          (128)
│    │    └─ReLU: 3-3                    [64, 64, 56, 56]          --
│    │    └─Conv2d: 3-4                  [64, 64, 56, 56]          (36,864)
│    │    └─BatchNorm2d: 3-5             [64, 64, 56, 56]          (128)
│    │    └─ReLU: 3-6                    [64, 64, 56, 56]   

In [34]:

loss_func = nn.CrossEntropyLoss()  #applied creoss entropy loss

learning_rate = 0.09
optimizer = optim.Adam(resnet50.parameters(), lr=learning_rate)

In [35]:
def train(model, train_loader):
    model.train()
    model.to(device)

    running_loss = 0
    correct_predictions = 0
    total_train_samples = 0

    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, dim=1)
        total_train_samples += labels.shape[0]
        correct_predictions += (predicted == labels).sum().item()

    train_avg_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct_predictions / total_train_samples
    return train_avg_loss, train_accuracy

In [36]:
def main(model, train_loader):

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    best_val_acc = 0.0
    best_weights = None

    for epoch in range(num_of_epoch):
        train_loss, train_accuracy = train(model, train_loader)


        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        print(f"Epoch {epoch+1:0>2}/{num_of_epoch} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%")

        # Logging metrics to tensorboard
        writer.add_scalar('Loss/train', train_loss)
        writer.add_scalar('Loss/val', val_loss)
        writer.add_scalar('Accuracy/train', train_accuracy)
        writer.add_scalar('Accuracy/val', val_accuracy)

        if train_accuracy > best_val_acc:
            best_val_acc = train_accuracy
            best_weights =  model.state_dict()
            print(f"Saving best model...💾")
            torch.save(best_weights, "best.pt")

    return train_losses, train_accuracies

In [37]:
train_losses, train_accuracies = main(resnet50, train_data_loader)

Training:   5%|▌         | 13/250 [02:03<37:37,  9.53s/it]


KeyboardInterrupt: 